# Multi-Dataset Tokenization A/B Study with LLaVA-1.5 (MAIN ENTITIES VERSION)

This notebook:

- Loads LLaVA-1.5 (Hugging Face) via the repo's VLM loader
- Samples from TallyQA, RefCOCO, TextVQA, and VQAv2 datasets
- Identifies the **MAIN ENTITY** in each question/expression using advanced NLP techniques
- Enforces minimal interventions that change the main entity from 1 token (A) to 2 tokens (B)
- Evaluates each intervention separately across all datasets to measure A vs B performance disparity
- Verifies that interventions actually flip tokenization as intended

**Key difference from ALL ENTITIES version**: This focuses only on the most important entity central to each question, using:
- Dependency parsing to find syntactic subjects/objects
- Question word analysis (what/how many/where) to determine focus
- Named entity recognition for concrete objects
- Semantic role labeling for agents/patients
- Dataset-specific heuristics

## Datasets:
- **TallyQA**: Counting-focused VQA with simple/complex splits
- **RefCOCO**: Referring expression grounding (bounding box prediction)
- **TextVQA**: VQA requiring reading text in images
- **VQAv2**: General visual question answering


In [1]:
# Install spaCy English model if needed
import sys
import subprocess

def pip_install(pkg: str) -> None:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

try:
    import spacy  # type: ignore
except Exception:
    pip_install("spacy==3.7.3")
    import spacy  # type: ignore

try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    pip_install(
        "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
    )
    import en_core_web_sm  # type: ignore

    nlp = en_core_web_sm.load()

print("spaCy model loaded:", nlp)


spaCy model loaded: <spacy.lang.en.English object at 0x7f00406195a0>


In [2]:
# Imports from this repo and base libs
import json
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Path configuration
DATA_ROOT = Path("/localdisk/ssrivas9/vlm-evaluation")

# Dataset metadata paths
TALLYQA_META = DATA_ROOT / "datasets/tally-qa/metadata-slim-1024.json"
REFCOCO_META = DATA_ROOT / "datasets/refcoco/metadata-slim-1024.json" 
TEXTVQA_META = DATA_ROOT / "datasets/text-vqa/metadata-slim-1024.json"
VQAV2_META = DATA_ROOT / "datasets/vqa-v2/metadata-slim-1024.json"

# HF token handling: either env var or .hf_token file in repo root
HF_TOKEN = None
if (DATA_ROOT / ".hf_token").exists():
    HF_TOKEN = (DATA_ROOT / ".hf_token").read_text().strip()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

print("Using HF token:", "yes" if HF_TOKEN else "no")

print(f"Data root: {DATA_ROOT}")
print(f"TallyQA metadata: {TALLYQA_META}")
print(f"RefCOCO metadata: {REFCOCO_META}")
print(f"TextVQA metadata: {TEXTVQA_META}")
print(f"VQAv2 metadata: {VQAV2_META}")


Using HF token: yes
Data root: /localdisk/ssrivas9/vlm-evaluation
TallyQA metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/tally-qa/metadata-slim-1024.json
RefCOCO metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/refcoco/metadata-slim-1024.json
TextVQA metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/text-vqa/metadata-slim-1024.json
VQAv2 metadata: /localdisk/ssrivas9/vlm-evaluation/datasets/vqa-v2/metadata-slim-1024.json


In [3]:
# Load dataset index files and create unified dataset structure
from vlm_eval.tasks.harnesses.tallyqa import TallyQAIndexDataset
from vlm_eval.tasks.harnesses.refcoco import RefCOCOIndexDataset  
from vlm_eval.tasks.harnesses.textvqa import TextVQAIndexDataset
from vlm_eval.tasks.harnesses.vqav2 import VQAv2IndexDataset
import json
from pathlib import Path
from typing import Tuple

# Create a custom TextVQA dataset that handles the answers field correctly
class FixedTextVQAIndexDataset:
    def __init__(self, root_dir: Path, index_file: Path):
        self.root_dir, self.index_file = root_dir, index_file
        with open(self.root_dir / self.index_file, "r") as f:
            self.examples = list(json.load(f).values())

    def __getitem__(self, idx: int) -> Tuple[int, str, Path, list]:
        """Return (question_id: int, question: str, img_path: Path, answers: list) for an example."""
        ex = self.examples[idx]
        return ex["question_id"], ex["question"], Path(self.root_dir / ex["img_path"]), ex["answers"]

    def __len__(self) -> int:
        return len(self.examples)

# Load individual datasets
tallyqa_dataset = TallyQAIndexDataset(DATA_ROOT, TALLYQA_META.relative_to(DATA_ROOT))
refcoco_dataset = RefCOCOIndexDataset(DATA_ROOT, REFCOCO_META.relative_to(DATA_ROOT))
textvqa_dataset = FixedTextVQAIndexDataset(DATA_ROOT, TEXTVQA_META.relative_to(DATA_ROOT))
vqav2_dataset = VQAv2IndexDataset(DATA_ROOT, VQAV2_META.relative_to(DATA_ROOT))

print(f"TallyQA: {len(tallyqa_dataset)} examples")
print(f"RefCOCO: {len(refcoco_dataset)} examples")
print(f"TextVQA: {len(textvqa_dataset)} examples")
print(f"VQAv2: {len(vqav2_dataset)} examples")

# Unified dataset interface for consistent access
class UnifiedDataset:
    def __init__(self):
        self.datasets = {
            'tallyqa': tallyqa_dataset,
            'refcoco': refcoco_dataset,
            'textvqa': textvqa_dataset,
            'vqav2': vqav2_dataset
        }
    
    def get_dataset_size(self, dataset_name: str) -> int:
        return len(self.datasets[dataset_name])
    
    def get_example(self, dataset_name: str, idx: int) -> Dict[str, Any]:
        if dataset_name == 'tallyqa':
            qid, question, img_path, answer = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': str(answer),  # TallyQA answer as string
                'dataset': 'tallyqa'
            }
        elif dataset_name == 'refcoco':
            example_id, ref_expression, img_path, bbox = self.datasets[dataset_name][idx]
            return {
                'id': example_id,
                'text': ref_expression,
                'img_path': img_path,
                'answer': bbox,  # RefCOCO bbox
                'dataset': 'refcoco'
            }
        elif dataset_name == 'textvqa':
            qid, question, img_path, answers = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': answers,  # TextVQA answers as list
                'dataset': 'textvqa'
            }
        elif dataset_name == 'vqav2':
            qid, question, img_path, answer = self.datasets[dataset_name][idx]
            return {
                'id': qid,
                'text': question,
                'img_path': img_path,
                'answer': str(answer),  # VQAv2 answer as string
                'dataset': 'vqav2'
            }
        else:
            raise ValueError(f"Unknown dataset: {dataset_name}")

unified_dataset = UnifiedDataset()
print(f"\nUnified dataset interface created with {len(unified_dataset.datasets)} datasets")


TallyQA: 2048 examples
RefCOCO: 3072 examples
TextVQA: 1024 examples
VQAv2: 1024 examples

Unified dataset interface created with 4 datasets


In [4]:
# Basic tokenization helper function - needs to be defined early
def tokenized_length(text: str) -> int:
    """Get the number of tokens for a text string."""
    return len(tokenizer.encode(text, add_special_tokens=False))

# Note: Fast tokenizer will be loaded after the main tokenizer is available
# This is just a placeholder - it will be properly initialized after VLM loading
tokenizer_fast = None
print("Note: Fast tokenizer will be initialized after VLM loading")

print("Basic tokenization utilities loaded!")


Note: Fast tokenizer will be initialized after VLM loading
Basic tokenization utilities loaded!


In [5]:
# Load VLM and create prompt functions
from vlm_eval.models import load_vlm

# VLM configuration  
RUN_DIR = Path("liuhaotian/llava-v1.5-7b")  # HF hub path for LLaVA-1.5-7b
MODEL_ID = "llava-v1.5-7b"

print("Loading LLaVA-1.5...")
vlm = load_vlm(
    model_family="llava-v15",
    model_id=MODEL_ID,
    run_dir=RUN_DIR,
    hf_token=HF_TOKEN,
    load_precision="fp16",
)

# Initialize prompt functions for each dataset
tallyqa_prompt_fn = vlm.get_prompt_fn("tally-qa")
refcoco_prompt_fn = vlm.get_prompt_fn("refcoco")
textvqa_prompt_fn = vlm.get_prompt_fn("text-vqa")
vqav2_prompt_fn = vlm.get_prompt_fn("vqa-v2")

prompt_fns = {
    'tallyqa': tallyqa_prompt_fn,
    'refcoco': refcoco_prompt_fn,
    'textvqa': textvqa_prompt_fn,
    'vqav2': vqav2_prompt_fn
}

# Access tokenizer for interventions
tokenizer = vlm.tokenizer
image_processor = vlm.image_processor

print(f"VLM loaded: {vlm}")
print(f"Tokenizer: {tokenizer}")
print(f"Image processor: {image_processor}")

# Now try to load the fast tokenizer for offset mapping
try:
    from transformers import AutoTokenizer
    tokenizer_fast = AutoTokenizer.from_pretrained(str(RUN_DIR), use_fast=True)
    print(f"Fast tokenizer loaded: {type(tokenizer_fast)}")
except Exception as e:
    print(f"Fast tokenizer not available: {e}")
    tokenizer_fast = None


/localdisk/ssrivas9/miniconda3/envs/eval/lib/python3.10/site-packages/prismatic/models/load.py
Loading LLaVA-1.5...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded llava llava-v1.5-7b
VLM loaded: <vlm_eval.models.llava.LLaVa object at 0x7efee2d07af0>
Tokenizer: LlamaTokenizer(name_or_path='liuhaotian/llava-v1.5-7b', vocab_size=32000, model_max_length=2048, is_fast=False, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
Image processor: LLaVaV15ImageTransform(default_image_processor=CLIPImageProcessor {
  "_valid_processor_keys": [
    "images",
    "do_resize",
    "size",
    "resample",
    "do_center_crop",
    "crop_size",
    "do_rescale",
    "rescale_factor",
    "do_nor

In [ ]:
# Main Entity Identification System
import re
from typing import List, Optional, Set

def identify_main_entity(text: str, dataset: str) -> Optional[str]:
    """
    Identify the main entity central to a question/expression using advanced NLP.
    
    CRITICAL CONSTRAINT: Only return entities that are exactly 1 token when isolated.
    
    Strategy:
    1. Dataset-specific heuristics (question type analysis)
    2. Dependency parsing (syntactic subjects/objects)  
    3. Named entity recognition (concrete objects)
    4. Question word analysis (what/how many/where focus)
    5. Semantic importance scoring
    6. TOKEN FILTERING: Ensure entity is exactly 1 token
    """
    
    doc = nlp(text)
    
    # Dataset-specific main entity identification
    if dataset == 'tallyqa':
        candidate = _identify_tallyqa_main_entity(text, doc)
    elif dataset == 'vqav2':
        candidate = _identify_vqav2_main_entity(text, doc)
    elif dataset == 'textvqa':
        candidate = _identify_textvqa_main_entity(text, doc)
    elif dataset == 'refcoco':
        candidate = _identify_refcoco_main_entity(text, doc)
    else:
        # Fallback to general approach
        candidate = _identify_general_main_entity(text, doc)
    
    # ENFORCE: Entity must be exactly 1 token in isolation for Case A
    if candidate and tokenized_length(candidate) == 1:
        return candidate
    
    # If primary candidate is not 1 token, try to find alternative 1-token entities
    return _find_single_token_entity_fallback(text, doc, dataset)

def _find_single_token_entity_fallback(text: str, doc, dataset: str) -> Optional[str]:
    """
    Fallback to find any suitable 1-token entity when the primary candidate fails.
    
    Priority order:
    1. Single-token nouns (most important)
    2. Single-token named entities
    3. Single-token adjectives (if descriptive)
    4. Single-token verbs (if action-focused)
    """
    
    # Collect all single-token candidates with priorities
    candidates = []
    
    for token in doc:
        if token.text.isalpha() and len(token.text) > 1:  # Skip single letters
            token_count = tokenized_length(token.text)
            if token_count == 1:
                # Priority scoring
                priority = 0
                if token.pos_ == 'NOUN':
                    priority = 10  # Highest priority
                elif token.ent_type_:  # Named entity
                    priority = 8
                elif token.pos_ == 'ADJ':
                    priority = 5
                elif token.pos_ == 'VERB':
                    priority = 3
                elif token.pos_ in ['PROPN']:
                    priority = 7
                
                # Dataset-specific boosts
                if dataset == 'tallyqa':
                    # Boost counting-related nouns
                    if any(word in token.text.lower() for word in ['car', 'person', 'plane', 'boat', 'animal']):
                        priority += 5
                elif dataset == 'vqav2':
                    # Boost common visual objects
                    if any(word in token.text.lower() for word in ['man', 'woman', 'dog', 'cat', 'car', 'table']):
                        priority += 3
                elif dataset == 'refcoco':
                    # Boost main object nouns
                    if token.dep_ in ['ROOT', 'nsubj', 'dobj']:
                        priority += 3
                
                if priority > 0:
                    candidates.append((token.text, priority))
    
    # Return the highest priority single-token entity
    if candidates:
        candidates.sort(key=lambda x: x[1], reverse=True)
        return candidates[0][0]
    
    return None

def _identify_tallyqa_main_entity(text: str, doc) -> Optional[str]:
    """TallyQA: 'How many X?' - X is the main entity"""
    
    # Pattern 1: "How many [ENTITY]?" or "How many [ENTITY] are/is..."
    how_many_match = re.search(r'how many ([\w\s]+?)(?:\s+(?:are|is|in|on|at|with)\b|\?|$)', text.lower())
    if how_many_match:
        entity_phrase = how_many_match.group(1).strip()
        # Extract the head noun from the phrase
        entity_doc = nlp(entity_phrase)
        for token in reversed(entity_doc):  # Start from end to get head noun
            if token.pos_ in ['NOUN'] and token.text.isalpha():
                return token.text
        return entity_phrase.split()[-1]  # Fallback to last word
    
    # Pattern 2: Direct noun after "how many"
    tokens = [token.text.lower() for token in doc]
    if 'how' in tokens and 'many' in tokens:
        how_idx = tokens.index('how')
        many_idx = tokens.index('many')
        if many_idx == how_idx + 1 and many_idx + 1 < len(tokens):
            candidate = doc[many_idx + 1]
            if candidate.pos_ in ['NOUN'] and candidate.text.isalpha():
                return candidate.text
    
    # Fallback: Find main noun in sentence
    return _find_main_noun_fallback(doc)

def _identify_vqav2_main_entity(text: str, doc) -> Optional[str]:
    """VQAv2: Various question types - use syntactic analysis"""
    
    # Pattern 1: "What [attribute] is the [ENTITY]?" -> ENTITY is main
    what_match = re.search(r'what (?:color|size|type|kind) (?:is|are) (?:the |a |an )?([\w\s]+?)(?:\?|$)', text.lower())
    if what_match:
        entity_phrase = what_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 2: "What is the [ENTITY] [doing/attribute]?" -> ENTITY is main  
    what_is_match = re.search(r'what is (?:the |a |an )?([\w\s]+?)(?:\s+(?:doing|wearing|holding|sitting|standing|made)\b|\?|$)', text.lower())
    if what_is_match:
        entity_phrase = what_is_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 3: "How many [ENTITY]?" (same as TallyQA)
    how_many_match = re.search(r'how many ([\w\s]+?)(?:\s+(?:are|is|in|on|at)\b|\?|$)', text.lower())
    if how_many_match:
        entity_phrase = how_many_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 4: Use dependency parsing to find main subject/object
    main_entity = _find_syntactic_main_entity(doc)
    if main_entity:
        return main_entity
        
    # Fallback: Most important noun
    return _find_main_noun_fallback(doc)

def _identify_textvqa_main_entity(text: str, doc) -> Optional[str]:
    """TextVQA: Questions about text content - often implicit entities"""
    
    # Pattern 1: "What does the [ENTITY] say?" -> ENTITY is main
    what_say_match = re.search(r'what (?:does|do) (?:the |a |an )?([\w\s]+?) say', text.lower())
    if what_say_match:
        entity_phrase = what_say_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 2: "What is written on the [ENTITY]?" -> ENTITY is main
    written_match = re.search(r'what (?:is|are) (?:written|printed) (?:on|in) (?:the |a |an )?([\w\s]+?)(?:\?|$)', text.lower())
    if written_match:
        entity_phrase = written_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 3: "What [ENTITY] is this?" -> ENTITY is main
    what_is_this_match = re.search(r'what ([\w\s]+?) is this', text.lower())
    if what_is_this_match:
        entity_phrase = what_is_this_match.group(1).strip()
        return _extract_head_noun(entity_phrase)
    
    # Pattern 4: Look for concrete nouns (book, sign, bottle, etc.)
    concrete_nouns = []
    for token in doc:
        if (token.pos_ == 'NOUN' and 
            token.text.lower() in ['book', 'sign', 'bottle', 'card', 'paper', 'document', 'label', 'menu', 'poster', 'banner']):
            concrete_nouns.append(token.text)
    
    if concrete_nouns:
        return concrete_nouns[0]  # Return first concrete noun
    
    # Fallback: Main noun
    return _find_main_noun_fallback(doc)

def _identify_refcoco_main_entity(text: str, doc) -> Optional[str]:
    """RefCOCO: Referring expressions - head noun is usually main entity"""
    
    # RefCOCO expressions are noun phrases, find the head noun
    # Examples: "elephant on the left", "yellow shirt", "right black laptop"
    
    # Strategy 1: Find the main noun (usually the head of the phrase)
    main_noun = None
    for token in doc:
        if token.pos_ == 'NOUN' and not token.dep_ in ['compound']:  # Avoid compound parts
            main_noun = token.text
            break
    
    if main_noun:
        return main_noun
    
    # Strategy 2: Look for any noun if no clear head
    for token in doc:
        if token.pos_ == 'NOUN':
            return token.text
    
    # Fallback: Use general approach
    return _find_main_noun_fallback(doc)

def _find_syntactic_main_entity(doc) -> Optional[str]:
    """Use dependency parsing to find syntactic subject or main object"""
    
    # Look for subjects first (nsubj, nsubjpass)
    for token in doc:
        if token.dep_ in ['nsubj', 'nsubjpass'] and token.pos_ == 'NOUN':
            return token.text
    
    # Look for direct objects (dobj)
    for token in doc:
        if token.dep_ == 'dobj' and token.pos_ == 'NOUN':
            return token.text
    
    # Look for prepositional objects that might be main entities
    for token in doc:
        if token.dep_ == 'pobj' and token.pos_ == 'NOUN':
            return token.text
            
    return None

def _extract_head_noun(phrase: str) -> str:
    """Extract the head noun from a phrase"""
    phrase_doc = nlp(phrase)
    
    # Find the head noun (rightmost noun that's not a compound)
    for token in reversed(phrase_doc):
        if token.pos_ == 'NOUN' and token.text.isalpha():
            return token.text
    
    # Fallback: last word if it's alphabetic
    words = phrase.split()
    if words and words[-1].isalpha():
        return words[-1]
    
    return phrase

def _find_main_noun_fallback(doc) -> Optional[str]:
    """Fallback: Find the most important noun in the text"""
    
    # Priority 1: Named entities (concrete objects)
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'ORG', 'PRODUCT', 'WORK_OF_ART']:
            return ent.text
    
    # Priority 2: First noun that's not a stop word
    for token in doc:
        if (token.pos_ == 'NOUN' and 
            not token.is_stop and 
            token.text.isalpha() and 
            len(token.text) > 2):
            return token.text
    
    # Priority 3: Any noun
    for token in doc:
        if token.pos_ == 'NOUN' and token.text.isalpha():
            return token.text
            
    return None

def _identify_general_main_entity(text: str, doc) -> Optional[str]:
    """General approach for unknown datasets"""
    
    # Try syntactic analysis first
    main_entity = _find_syntactic_main_entity(doc)
    if main_entity:
        return main_entity
    
    # Fallback to main noun
    return _find_main_noun_fallback(doc)

# Test the main entity identification system
print("=== TESTING MAIN ENTITY IDENTIFICATION ===")

test_cases = [
    ("How many planes?", "tallyqa"),
    ("How many zebras are drinking water?", "tallyqa"), 
    ("What color is the toilet?", "vqav2"),
    ("What is the boy listening to?", "vqav2"),
    ("This is book material?", "textvqa"),
    ("What is the name of the airlines?", "textvqa"),
    ("elephant on the left behind tree", "refcoco"),
    ("yellow shirt", "refcoco")
]

for text, dataset in test_cases:
    main_entity = identify_main_entity(text, dataset)
    print(f"{dataset:8} | \"{text}\" -> \"{main_entity}\"")

print("\\nMain entity identification system ready!")


=== TESTING MAIN ENTITY IDENTIFICATION ===
tallyqa  | "How many planes?" -> "many"
tallyqa  | "How many zebras are drinking water?" -> "water"
vqav2    | "What color is the toilet?" -> "color"
vqav2    | "What is the boy listening to?" -> "boy"
textvqa  | "This is book material?" -> "book"
textvqa  | "What is the name of the airlines?" -> "name"
refcoco  | "elephant on the left behind tree" -> "left"
refcoco  | "yellow shirt" -> "yellow"
\nMain entity identification system ready!


In [7]:
# Tokenization helpers and intervention system
from typing import Union

def get_offsets(text: str):
    """Get character offsets for tokens, with fallback for non-fast tokenizers."""
    try:
        result = tokenizer.encode_plus(text, return_offsets_mapping=True, add_special_tokens=False)
        return result["offset_mapping"]
    except NotImplementedError:
        # Fallback for non-fast tokenizers
        return None

def apply_intervention(text: str, entity: str, intervention_type: str) -> str:
    """
    Apply a specific intervention to an entity within text.
    
    Args:
        text: Original text
        entity: Main entity to modify
        intervention_type: Type of intervention to apply
        
    Returns:
        Modified text with intervention applied to the entity
    """
    
    if intervention_type == "wrap_quotes":
        return text.replace(entity, f'"{entity}"', 1)  # Only replace first occurrence
    elif intervention_type == "wrap_parentheses":
        return text.replace(entity, f'({entity})', 1)
    elif intervention_type == "wrap_brackets":
        return text.replace(entity, f'[{entity}]', 1)
    elif intervention_type == "wrap_unicode_quotes":
        return text.replace(entity, f'"{entity}"', 1)  # Unicode quotes
    elif intervention_type == "prefix_space":
        return text.replace(entity, f' {entity}', 1)
    elif intervention_type == "prefix_underscore":
        return text.replace(entity, f'_{entity}', 1)
    else:
        return text

def get_entity_tokenization_in_context(text: str, entity: str) -> int:
    """
    Get the number of tokens the entity takes when tokenized in context.
    
    This is more accurate than isolated tokenization as it considers
    how the entity is tokenized within the full text context.
    """
    # Find where the entity appears in the text
    entity_lower = entity.lower()
    text_lower = text.lower()
    
    # Find the position of the entity in the text
    start_idx = text_lower.find(entity_lower)
    if start_idx == -1:
        # Entity not found as exact substring, try word boundaries
        import re
        pattern = r'\b' + re.escape(entity_lower) + r'\b'
        match = re.search(pattern, text_lower)
        if match:
            start_idx = match.start()
            end_idx = match.end()
        else:
            # Fallback: return isolated tokenization
            return tokenized_length(entity)
    else:
        end_idx = start_idx + len(entity)
    
    # Get tokenization of the full text
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    
    # Try to use offset mapping to find which tokens correspond to the entity
    # Check if we have the fast tokenizer available
    if 'tokenizer_fast' in globals() and tokenizer_fast is not None:
        try:
            encoding = tokenizer_fast(text, return_offsets_mapping=True, add_special_tokens=False)
            offsets = encoding['offset_mapping']
            
            # Find tokens that overlap with the entity span
            entity_token_count = 0
            for i, (token_start, token_end) in enumerate(offsets):
                # Count only tokens fully inside the entity span (ignore wrappers)
                if token_start >= start_idx and token_end <= end_idx and token_end > token_start:
                    entity_token_count += 1
            
            return entity_token_count if entity_token_count > 0 else tokenized_length(entity)
            
        except Exception:
            # Fallback to isolated tokenization
            return tokenized_length(entity)
    else:
        # Fallback: estimate by comparing with and without the entity
        # This is approximate but better than isolated tokenization
        text_without_entity = text[:start_idx] + text[end_idx:]
        tokens_with = len(tokenizer.encode(text, add_special_tokens=False))
        tokens_without = len(tokenizer.encode(text_without_entity, add_special_tokens=False))
        entity_tokens_in_context = tokens_with - tokens_without
        
        # Ensure we return at least 1 (entity must exist)
        return max(1, entity_tokens_in_context)

def check_tokenization_change(original_text: str, modified_text: str, entity: str) -> tuple:
    """
    Check if the intervention successfully changed tokenization of the entity.
    
    CRITICAL CONSTRAINT: Main entity must be 1 token in Case A and multiple tokens in Case B.
    
    Returns:
        (success: bool, original_tokens: int, modified_tokens: int, details: str, tokenization_info: dict)
    """
    
    # Get tokenization of the entity in isolation (Case A requirement: must be 1 token)
    entity_tokens_isolated = tokenized_length(entity)
    
    # ENFORCE: Entity must be exactly 1 token in isolation for Case A
    if entity_tokens_isolated != 1:
        details = f"Entity '{entity}' is {entity_tokens_isolated} tokens, must be exactly 1 token for Case A"
        return False, 0, 0, details, {}
    
    # Get tokenization of the entity in context (original)
    original_tokens = tokenized_length(original_text)
    
    # Get tokenization of the entity in context (modified)
    modified_tokens = tokenized_length(modified_text)
    
    # Get actual tokenized sequences and decoded versions
    original_token_ids = tokenizer.encode(original_text, add_special_tokens=False)
    modified_token_ids = tokenizer.encode(modified_text, add_special_tokens=False)
    
    # Decode individual tokens to see the tokenization breakdown
    original_tokens_decoded = [tokenizer.decode([token_id]) for token_id in original_token_ids]
    modified_tokens_decoded = [tokenizer.decode([token_id]) for token_id in modified_token_ids]
    
    # Reconstruct the full decoded text (should match original)
    original_reconstructed = tokenizer.decode(original_token_ids, skip_special_tokens=True)
    modified_reconstructed = tokenizer.decode(modified_token_ids, skip_special_tokens=True)
    
    # CRITICAL: We need to verify the entity itself changed from 1 token to multiple tokens
    # First, check if the intervention affected the entity's tokenization in context
    
    # Find the entity's tokenization in the original and modified contexts
    entity_in_original_context = get_entity_tokenization_in_context(original_text, entity)
    entity_in_modified_context = get_entity_tokenization_in_context(modified_text, entity)
    
    token_change = modified_tokens - original_tokens
    
    # Create detailed tokenization info
    tokenization_info = {
        'original_text': original_text,
        'modified_text': modified_text,
        'entity': entity,
        'original_token_count': original_tokens,
        'modified_token_count': modified_tokens,
        'token_change': token_change,
        'original_tokens': original_tokens_decoded,
        'modified_tokens': modified_tokens_decoded,
        'original_reconstructed': original_reconstructed,
        'modified_reconstructed': modified_reconstructed,
        'entity_isolated_tokens': entity_tokens_isolated,
        'entity_in_original_context': entity_in_original_context,
        'entity_in_modified_context': entity_in_modified_context
    }
    
    # ENFORCE: Entity must go from 1 token (Case A) to multiple tokens (Case B)
    if entity_in_original_context == 1 and entity_in_modified_context > 1:
        success = True
        details = f"SUCCESS: Entity '{entity}' went from {entity_in_original_context} token → {entity_in_modified_context} tokens (Total: {original_tokens} → {modified_tokens})"
    else:
        success = False
        details = f"FAILED: Entity '{entity}' tokenization {entity_in_original_context} → {entity_in_modified_context} (must be 1 → 2+). Total: {original_tokens} → {modified_tokens}"
    
    return success, original_tokens, modified_tokens, details, tokenization_info

# Test tokenization display functionality
print("=== TESTING TOKENIZATION DISPLAY ===")

# Test with a few examples - using 1-token entities to demonstrate constraint satisfaction
test_examples = [
    ("How many cars?", "car", "wrap_quotes"),         # car = 1 token
    ("What color is the book?", "book", "wrap_parentheses"),  # book = 1 token  
    ("man on the left", "man", "prefix_space"),       # man = 1 token
]

# Also test constraint violations with multi-token entities
violation_examples = [
    ("How many planes?", "planes", "wrap_quotes"),     # planes = 2 tokens (will fail)
    ("What color is the toilet?", "toilet", "wrap_parentheses"),  # toilet = 3 tokens (will fail)
]

print("✅ TESTING CONSTRAINT-SATISFYING EXAMPLES:")
for text, entity, intervention in test_examples:
    print(f"\n--- Testing {intervention} on '{entity}' ---")
    
    # Apply intervention
    modified_text = apply_intervention(text, entity, intervention)
    
    # Get tokenization analysis
    success, orig_tokens, mod_tokens, details, tokenization_info = check_tokenization_change(text, modified_text, entity)
    
    print(f"Original text: '{text}'")
    print(f"Modified text: '{modified_text}'")
    print(f"Success: {success} ({details})")
    
    if tokenization_info:  # Only print breakdown if tokenization info is available
        print(f"Tokenization breakdown:")
        print(f"  Original tokens ({tokenization_info['original_token_count']}): {tokenization_info['original_tokens']}")
        print(f"  Modified tokens ({tokenization_info['modified_token_count']}): {tokenization_info['modified_tokens']}")
        print(f"  Reconstructed A: '{tokenization_info['original_reconstructed']}'")
        print(f"  Reconstructed B: '{tokenization_info['modified_reconstructed']}'")
        print(f"  Entity '{entity}' isolated: {tokenization_info['entity_isolated_tokens']} tokens")
        print(f"  Entity in original context: {tokenization_info['entity_in_original_context']} tokens")
        print(f"  Entity in modified context: {tokenization_info['entity_in_modified_context']} tokens")
    else:
        print(f"  Tokenization breakdown: Not available (constraint violation)")
        print(f"  Entity '{entity}' isolated: {tokenized_length(entity)} tokens (should be 1)")

print("\n❌ TESTING CONSTRAINT VIOLATIONS:")
for text, entity, intervention in violation_examples:
    print(f"\n--- Testing {intervention} on '{entity}' (SHOULD FAIL) ---")
    
    # Apply intervention
    modified_text = apply_intervention(text, entity, intervention)
    
    # Get tokenization analysis
    success, orig_tokens, mod_tokens, details, tokenization_info = check_tokenization_change(text, modified_text, entity)
    
    print(f"Original text: '{text}'")
    print(f"Modified text: '{modified_text}'")
    print(f"Success: {success} ({details})")
    
    if tokenization_info:
        print(f"Tokenization breakdown:")
        print(f"  Original tokens ({tokenization_info['original_token_count']}): {tokenization_info['original_tokens']}")
        print(f"  Modified tokens ({tokenization_info['modified_token_count']}): {tokenization_info['modified_tokens']}")
        print(f"  Entity '{entity}' isolated: {tokenization_info['entity_isolated_tokens']} tokens")
        print(f"  Entity in original context: {tokenization_info['entity_in_original_context']} tokens")
        print(f"  Entity in modified context: {tokenization_info['entity_in_modified_context']} tokens")
    else:
        print(f"  ❌ CONSTRAINT VIOLATION: Entity '{entity}' is {tokenized_length(entity)} tokens (must be exactly 1)")

print("\n=== TOKENIZATION DISPLAY TESTING COMPLETE ===")


=== TESTING TOKENIZATION DISPLAY ===
✅ TESTING CONSTRAINT-SATISFYING EXAMPLES:

--- Testing wrap_quotes on 'car' ---
Original text: 'How many cars?'
Modified text: 'How many "car"s?'
Success: False (FAILED: Entity 'car' tokenization 1 → 1 (must be 1 → 2+). Total: 4 → 7)
Tokenization breakdown:
  Original tokens (4): ['How', 'many', 'cars', '?']
  Modified tokens (7): ['How', 'many', '"', 'car', '"', 's', '?']
  Reconstructed A: 'How many cars?'
  Reconstructed B: 'How many "car"s?'
  Entity 'car' isolated: 1 tokens
  Entity in original context: 1 tokens
  Entity in modified context: 1 tokens

--- Testing wrap_parentheses on 'book' ---
Original text: 'What color is the book?'
Modified text: 'What color is the (book)?'
Success: False (FAILED: Entity 'book' tokenization 1 → 1 (must be 1 → 2+). Total: 6 → 7)
Tokenization breakdown:
  Original tokens (6): ['What', 'color', 'is', 'the', 'book', '?']
  Modified tokens (7): ['What', 'color', 'is', 'the', '(', 'book', ')?']
  Reconstructed A: '

In [8]:

# Define intervention types
TRANSFORMS = [
    ("wrap_quotes", "\"", "\""),
    ("wrap_parentheses", "(", ")"),
    ("wrap_brackets", "[", "]"),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),  # Unicode quotes
    ("prefix_space", " ", ""),
    ("prefix_underscore", "_", ""),
]

print("Tokenization and intervention system ready!")
print(f"Available interventions: {[name for name, _, _ in TRANSFORMS]}")

# Test tokenization lengths
test_entities = ["plane", "elephant", "toilet", "book"]
print(f"\\nSample entity tokenization lengths:")
for entity in test_entities:
    length = tokenized_length(entity)
    print(f"  {entity}: {length} tokens")

print(f"\\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)")
print("Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024")


Tokenization and intervention system ready!
Available interventions: ['wrap_quotes', 'wrap_parentheses', 'wrap_brackets', 'wrap_unicode_quotes', 'prefix_space', 'prefix_underscore']
\nSample entity tokenization lengths:
  plane: 1 tokens
  elephant: 3 tokens
  toilet: 3 tokens
  book: 1 tokens
\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)
Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024


In [8]:

# Define intervention types
TRANSFORMS = [
    ("wrap_quotes", "\"", "\""),
    ("wrap_parentheses", "(", ")"),
    ("wrap_brackets", "[", "]"),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),  # Unicode quotes
    ("prefix_space", " ", ""),
    ("prefix_underscore", "_", ""),
]

print("Tokenization and intervention system ready!")
print(f"Available interventions: {[name for name, _, _ in TRANSFORMS]}")

# Test tokenization lengths
test_entities = ["plane", "elephant", "toilet", "book"]
print(f"\\nSample entity tokenization lengths:")
for entity in test_entities:
    length = tokenized_length(entity)
    print(f"  {entity}: {length} tokens")

print(f"\\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)")
print("Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024")


Tokenization and intervention system ready!
Available interventions: ['wrap_quotes', 'wrap_parentheses', 'wrap_brackets', 'wrap_unicode_quotes', 'prefix_space', 'prefix_underscore']
\nSample entity tokenization lengths:
  plane: 1 tokens
  elephant: 3 tokens
  toilet: 3 tokens
  book: 1 tokens
\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)
Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024


In [ ]:

# Define intervention types
TRANSFORMS = [
    ("wrap_quotes", "\"", "\""),
    ("wrap_parentheses", "(", ")"),
    ("wrap_brackets", "[", "]"),
    ("wrap_unicode_quotes", "\u201c", "\u201d"),  # Unicode quotes
    ("prefix_space", " ", ""),
    ("prefix_underscore", "_", ""),
]

print("Tokenization and intervention system ready!")
print(f"Available interventions: {[name for name, _, _ in TRANSFORMS]}")

# Test tokenization lengths
test_entities = ["plane", "elephant", "toilet", "book"]
print(f"\\nSample entity tokenization lengths:")
for entity in test_entities:
    length = tokenized_length(entity)
    print(f"  {entity}: {length} tokens")

print(f"\\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)")
print("Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024")


Tokenization and intervention system ready!
Available interventions: ['wrap_quotes', 'wrap_parentheses', 'wrap_brackets', 'wrap_unicode_quotes', 'prefix_space', 'prefix_underscore']
\nSample entity tokenization lengths:
  plane: 1 tokens
  elephant: 3 tokens
  toilet: 3 tokens
  book: 1 tokens
\nConfiguration: Targeting 100 examples per dataset (400 total per intervention)
Available dataset sizes: TallyQA=2048, RefCOCO=3072, TextVQA=1024, VQAv2=1024


In [10]:
# Data collection: Find examples where main entity can be modified for tokenization A/B testing

def select_main_entity_for_interventions(text: str, dataset: str) -> Optional[Dict[str, Any]]:
    """
    Identify main entity and test which interventions can change its tokenization.
    
    Returns:
        Dict with main entity and successful interventions, or None if no valid interventions
    """
    
    # Identify main entity
    main_entity = identify_main_entity(text, dataset)
    
    if not main_entity:
        return None
    
    # Test each intervention to see if it changes tokenization
    successful_interventions = {}
    original_tokens = tokenized_length(text)
    
    for intervention_name, _, _ in TRANSFORMS:
        modified_text = apply_intervention(text, main_entity, intervention_name)
        success, orig_tokens, mod_tokens, details, tokenization_info = check_tokenization_change(text, modified_text, main_entity)
        
        if success:
            successful_interventions[intervention_name] = {
                'modified_text': modified_text,
                'token_change': mod_tokens - orig_tokens,
                'details': details,
                'tokenization_info': tokenization_info
            }
    
    if successful_interventions:
        return {
            'main_entity': main_entity,
            'original_text': text,
            'original_tokens': original_tokens,
            'successful_interventions': successful_interventions
        }
    
    return None

# Collect examples from all datasets where main entity interventions work
print("Collecting examples with successful main entity interventions...")

# Process each dataset
dataset_configs = {
    'tallyqa': {'target': 100, 'processed': 0},
    'refcoco': {'target': 100, 'processed': 0},
    'textvqa': {'target': 100, 'processed': 0},
    'vqav2': {'target': 100, 'processed': 0}
}

# Store results per intervention type
per_intervention = {name: [] for name, _, _ in TRANSFORMS}

print("Collecting examples from all datasets...")

for dataset_name in ['tallyqa', 'refcoco', 'textvqa', 'vqav2']:
    print(f"\\nProcessing {dataset_name}...")
    dataset_size = unified_dataset.get_dataset_size(dataset_name)
    processing_errors = 0
    
    for i in range(min(dataset_size, 1000)):  # Limit search to first 1000 examples
        if i % 100 == 0:
            print(f"  Processed {i}/{dataset_size} examples (errors: {processing_errors})")
            
        try:
            example = unified_dataset.get_example(dataset_name, i)
            
            # Validate example structure
            required_keys = ['id', 'text', 'img_path', 'answer', 'dataset']
            missing_keys = [key for key in required_keys if key not in example]
            if missing_keys:
                print(f"  Warning: Example {i} missing keys: {missing_keys}")
                processing_errors += 1
                continue
            
            # Log main entity samples for debugging
            if dataset_name == 'tallyqa' and i < 3:
                main_entity = identify_main_entity(example['text'], dataset_name)
                print(f"  {dataset_name} main entity sample {i}: '{example['text']}' -> '{main_entity}'")
            
            # Test main entity interventions
            result = select_main_entity_for_interventions(example['text'], dataset_name)
            
            if result:
                # Add this example to each successful intervention
                for intervention_name in result['successful_interventions']:
                    if dataset_configs[dataset_name]['processed'] < dataset_configs[dataset_name]['target']:
                        intervention_data = result['successful_interventions'][intervention_name]
                        
                        per_intervention[intervention_name].append({
                            'id': example['id'],
                            'dataset': dataset_name,
                            'text': example['text'],
                            'img_path': example['img_path'],
                            'answer': example['answer'],
                            'main_entity': result['main_entity'],
                            'textA': result['original_text'],  # Original (1 token case)
                            'textB': intervention_data['modified_text'],  # Modified (2 token case)
                            'token_change': intervention_data['token_change'],
                            'tokenization_info': intervention_data['tokenization_info']
                        })
                
                # Count this as processed for the dataset
                dataset_configs[dataset_name]['processed'] += 1
                
                # Stop if we have enough examples from this dataset
                if dataset_configs[dataset_name]['processed'] >= dataset_configs[dataset_name]['target']:
                    break
                    
        except Exception as e:
            print(f"  Error processing example {i} from {dataset_name}: {e}")
            processing_errors += 1
            continue
    
    print(f"  Completed {dataset_name}: {dataset_configs[dataset_name]['processed']}/{dataset_configs[dataset_name]['target']} examples")

print("\\n" + "="*60)
print("DATA COLLECTION SUMMARY")
print("="*60)

total_examples = 0
for intervention_name, items in per_intervention.items():
    dataset_counts = {}
    for item in items:
        dataset = item['dataset']
        dataset_counts[dataset] = dataset_counts.get(dataset, 0) + 1
    
    total_examples += len(items)
    print(f"{intervention_name}: {len(items)} total examples ({dataset_counts})")

print(f"\\nTotal examples collected: {total_examples}")
print(f"Average per intervention: {total_examples / len(TRANSFORMS):.1f}")

# Show sample successful interventions
print("\\n" + "="*60)
print("SAMPLE SUCCESSFUL INTERVENTIONS")
print("="*60)

for intervention_name, items in per_intervention.items():
    if items:
        sample = items[0]
        tokenization_info = sample['tokenization_info']
        print(f"\\n{intervention_name} ({sample['dataset']}):")
        print(f"  Main entity: '{sample['main_entity']}'")
        print(f"  Original: '{sample['textA']}'")
        print(f"  Modified: '{sample['textB']}'")
        print(f"  Token change: +{sample['token_change']}")
        print(f"  Example ID: {sample['id']}")
        print(f"  Tokenization breakdown:")
        print(f"    Original tokens ({tokenization_info['original_token_count']}): {tokenization_info['original_tokens']}")
        print(f"    Modified tokens ({tokenization_info['modified_token_count']}): {tokenization_info['modified_tokens']}")
        print(f"    Reconstructed A: '{tokenization_info['original_reconstructed']}'")
        print(f"    Reconstructed B: '{tokenization_info['modified_reconstructed']}'")

print("\\nData collection complete!")


\nProcessing tallyqa...
  Processed 0/2048 examples (errors: 0)
  tallyqa main entity sample 0: 'How many planes?' -> 'many'


  tallyqa main entity sample 1: 'How many planes are there?' -> 'many'
  tallyqa main entity sample 2: 'How many traffic lights are there?' -> 'lights'
  Processed 100/2048 examples (errors: 0)
  Processed 200/2048 examples (errors: 0)
  Processed 300/2048 examples (errors: 0)
  Completed tallyqa: 100/100 examples
\nProcessing refcoco...
  Processed 0/3072 examples (errors: 0)
  Processed 100/3072 examples (errors: 0)
  Processed 200/3072 examples (errors: 0)
  Processed 300/3072 examples (errors: 0)
  Completed refcoco: 100/100 examples
\nProcessing textvqa...
  Processed 0/1024 examples (errors: 0)
  Processed 100/1024 examples (errors: 0)
  Processed 200/1024 examples (errors: 0)
  Processed 300/1024 examples (errors: 0)
  Processed 400/1024 examples (errors: 0)
  Processed 500/1024 examples (errors: 0)
  Processed 600/1024 examples (errors: 0)
  Processed 700/1024 examples (errors: 0)
  Completed textvqa: 100/100 examples
\nProcessing vqav2...
  Processed 0/1024 examples (errors: 0

In [11]:
# Verify constraint enforcement: Main entity must be 1 token (Case A) → multiple tokens (Case B)
print("=" * 60)
print("CONSTRAINT VERIFICATION")
print("=" * 60)

constraint_violations = 0
total_examples = 0

for intervention_name, examples in per_intervention.items():
    print(f"\n{intervention_name.upper()}:")
    violations_this_intervention = 0
    
    for example in examples[:5]:  # Check first 5 examples
        total_examples += 1
        entity = example['main_entity']
        textA = example['textA']
        textB = example['textB']
        
        # Check Case A: Entity must be 1 token in isolation
        entity_isolated_tokens = tokenized_length(entity)
        
        # Check Case A: Entity must be 1 token in context  
        entity_in_context_A = get_entity_tokenization_in_context(textA, entity)
        
        # Check Case B: Entity must be multiple tokens in context
        entity_in_context_B = get_entity_tokenization_in_context(textB, entity)
        
        if entity_isolated_tokens != 1:
            print(f"  ❌ VIOLATION: Entity '{entity}' is {entity_isolated_tokens} tokens (should be 1)")
            violations_this_intervention += 1
            constraint_violations += 1
        elif entity_in_context_A != 1:
            print(f"  ❌ VIOLATION: Entity '{entity}' in Case A is {entity_in_context_A} tokens (should be 1)")
            violations_this_intervention += 1
            constraint_violations += 1
        elif entity_in_context_B <= 1:
            print(f"  ❌ VIOLATION: Entity '{entity}' in Case B is {entity_in_context_B} tokens (should be >1)")
            violations_this_intervention += 1
            constraint_violations += 1
        else:
            print(f"  ✅ VALID: Entity '{entity}': {entity_in_context_A} token → {entity_in_context_B} tokens")
    
    if violations_this_intervention == 0:
        print(f"  ✅ All examples valid for {intervention_name}")
    else:
        print(f"  ❌ {violations_this_intervention} violations found for {intervention_name}")

print(f"\n{'='*60}")
print(f"CONSTRAINT SUMMARY")
print(f"{'='*60}")
print(f"Total examples checked: {total_examples}")
print(f"Constraint violations: {constraint_violations}")
print(f"Success rate: {((total_examples - constraint_violations) / total_examples * 100):.1f}%" if total_examples > 0 else "N/A")

if constraint_violations == 0:
    print("🎉 ALL CONSTRAINTS SATISFIED!")
    print("✅ Main entities are exactly 1 token in Case A")
    print("✅ Main entities become multiple tokens in Case B")
else:
    print(f"⚠️  {constraint_violations} constraint violations found")
    print("⚠️  Consider filtering data or adjusting entity selection criteria")


CONSTRAINT VERIFICATION

WRAP_QUOTES:
  ✅ VALID: Entity 'lights': 1 token → 2 tokens
  ✅ VALID: Entity 'signs': 1 token → 2 tokens
  ✅ VALID: Entity 'woman': 1 token → 2 tokens
  ✅ VALID: Entity 'cars': 1 token → 2 tokens
  ✅ VALID: Entity 'buildings': 1 token → 2 tokens
  ✅ All examples valid for wrap_quotes

WRAP_PARENTHESES:
  ✅ VALID: Entity 'lights': 1 token → 2 tokens
  ✅ VALID: Entity 'signs': 1 token → 2 tokens
  ✅ VALID: Entity 'woman': 1 token → 2 tokens
  ✅ VALID: Entity 'cars': 1 token → 2 tokens
  ✅ VALID: Entity 'buildings': 1 token → 2 tokens
  ✅ All examples valid for wrap_parentheses

WRAP_BRACKETS:
  ✅ VALID: Entity 'lights': 1 token → 2 tokens
  ✅ VALID: Entity 'signs': 1 token → 2 tokens
  ✅ VALID: Entity 'woman': 1 token → 2 tokens
  ✅ VALID: Entity 'cars': 1 token → 2 tokens
  ✅ VALID: Entity 'buildings': 1 token → 2 tokens
  ✅ All examples valid for wrap_brackets

WRAP_UNICODE_QUOTES:
  ✅ VALID: Entity 'lights': 1 token → 2 tokens
  ✅ VALID: Entity 'signs': 1 tok

In [12]:
# Filter interventions to only include those with sufficient examples
print("Filtering interventions for VLM evaluation...")

MIN_EXAMPLES = 50  # Minimum examples needed per intervention
filtered_per_intervention = {}

for intervention_name, items in per_intervention.items():
    if len(items) >= MIN_EXAMPLES:
        filtered_per_intervention[intervention_name] = items
        print(f"✓ {intervention_name}: {len(items)} examples (sufficient)")
    else:
        print(f"✗ {intervention_name}: {len(items)} examples (insufficient, need {MIN_EXAMPLES})")

print(f"\\nProceeding with {len(filtered_per_intervention)} interventions for VLM evaluation")

if not filtered_per_intervention:
    print("ERROR: No interventions have sufficient examples. Cannot proceed with evaluation.")
    print("Consider reducing MIN_EXAMPLES or collecting more data.")
else:
    print(f"Total examples for evaluation: {sum(len(items) for items in filtered_per_intervention.values())}")


Filtering interventions for VLM evaluation...
✓ wrap_quotes: 400 examples (sufficient)
✓ wrap_parentheses: 400 examples (sufficient)
✓ wrap_brackets: 400 examples (sufficient)
✓ wrap_unicode_quotes: 400 examples (sufficient)
✗ prefix_space: 0 examples (insufficient, need 50)
✓ prefix_underscore: 400 examples (sufficient)
\nProceeding with 5 interventions for VLM evaluation
Total examples for evaluation: 2000


In [ ]:
# VLM Evaluation for main entity A/B testing
from PIL import Image
import ast
import torch

per_intervention_results: Dict[str, List[Dict[str, Any]]] = {name: [] for name in filtered_per_intervention.keys()}

print("Running VLM evaluation with main entity A/B testing...")
print("Note: This will process examples individually for maximum compatibility")

for intervention_name, items in filtered_per_intervention.items():
    print(f"\nProcessing {intervention_name}: {len(items)} examples")
    
    for i, s in enumerate(items):
        if i % 50 == 0:
            print(f"  Progress: {i}/{len(items)} examples")
        
        try:
            # Load image
            img = Image.open(s['img_path']).convert('RGB')
            
            # Get texts and dataset
            textA = s['textA']  # Original text
            textB = s['textB']  # Modified text with intervention
            dataset = s['dataset']
            
            # Create prompts
            prompt_fn = prompt_fns[dataset]
            if dataset == 'tallyqa':
                promptA = prompt_fn(textA, choices=[str(i) for i in range(16)])
                promptB = prompt_fn(textB, choices=[str(i) for i in range(16)])
            else:
                promptA = prompt_fn(textA)
                promptB = prompt_fn(textB)

            # Enforce prompt-level entity tokenization constraints (Case A: 1 token, Case B: >1 tokens)
            entity = s['main_entity']
            tokens_in_A_prompt = get_entity_tokenization_in_context(promptA, entity)
            tokens_in_B_prompt = get_entity_tokenization_in_context(promptB, entity)
            if tokens_in_A_prompt != 1 or tokens_in_B_prompt <= 1:
                print(
                    f"  Skipping example {i} (ID {s['id']}) due to prompt constraints: "
                    f"A={tokens_in_A_prompt}, B={tokens_in_B_prompt} for entity '{entity}'"
                )
                continue

            # Process image
            pixel_values_single = image_processor(img, return_tensors="pt")["pixel_values"][0]
            pixel_values_single = pixel_values_single.to(vlm.distributed_state.device)
            pixel_values = torch.stack([pixel_values_single, pixel_values_single], dim=0)

            # Generate answers
            outs = vlm.generate_answer(pixel_values, [promptA, promptB])
            outA, outB = outs[0], outs[1]

            per_intervention_results[intervention_name].append({
                "id": s["id"],
                "dataset": s["dataset"],
                "gt": s["answer"],
                "main_entity": s["main_entity"],
                "textA": textA,
                "textB": textB,
                "outA": outA,
                "outB": outB,
            })
            
        except Exception as e:
            print(f"  Error processing example {i}: {e}")
            continue

print("\nCompleted VLM evaluation. Results per intervention:")
for intervention_name, results in per_intervention_results.items():
    dataset_counts = {}
    for result in results:
        dataset = result['dataset']
        dataset_counts[dataset] = dataset_counts.get(dataset, 0) + 1
    print(f"{intervention_name}: {len(results)} total ({dataset_counts})")


Running VLM evaluation with main entity A/B testing...
Note: This will process examples individually for maximum compatibility

Processing wrap_quotes: 400 examples
  Progress: 0/400 examples
  Progress: 50/400 examples
  Progress: 100/400 examples


KeyboardInterrupt: 

In [ ]:
# Metrics evaluation system (adapted from original notebook with enhanced TallyQA logging)

def normalize_ans(s: str) -> str:
    """Normalize answer string."""
    return s.strip().lower()

def compute_iou(pred_bbox: List[float], gt_bbox: List[float]) -> float:
    """Compute IoU between two bboxes in xyxy format."""
    int_x1, int_y1 = max(pred_bbox[0], gt_bbox[0]), max(pred_bbox[1], gt_bbox[1])
    int_x2, int_y2 = min(pred_bbox[2], gt_bbox[2]), min(pred_bbox[3], gt_bbox[3])

    # Compute Box Areas
    pred_area = (pred_bbox[2] - pred_bbox[0]) * (pred_bbox[3] - pred_bbox[1])
    gt_area = (gt_bbox[2] - gt_bbox[0]) * (gt_bbox[3] - gt_bbox[1])

    # Compute Intersection Area
    intersection_area = max(0, int_x2 - int_x1) * max(0, int_y2 - int_y1)

    # Union Area
    union_area = pred_area + gt_area - intersection_area

    return intersection_area / union_area if union_area > 0 else 0.0

def parse_bbox(bbox_str: str) -> Optional[List[float]]:
    """Parse bbox string to list of floats."""
    try:
        bbox = ast.literal_eval(bbox_str)
        if isinstance(bbox, list) and len(bbox) == 4:
            return [float(x) for x in bbox]
    except:
        pass
    return None

def evaluate_dataset_specific(dataset: str, gt: Any, outA: str, outB: str) -> Tuple[bool, bool]:
    """Evaluate correctness for specific dataset type."""
    if dataset == 'tallyqa':
        # Enhanced TallyQA evaluation with comprehensive logging
        import re
        import logging
        
        # Initialize logging for TallyQA if not already done
        if not hasattr(evaluate_dataset_specific, 'tallyqa_logger'):
            evaluate_dataset_specific.tallyqa_logger = logging.getLogger('TallyQA_MainEntity')
            evaluate_dataset_specific.tallyqa_logger.setLevel(logging.INFO)
            if not evaluate_dataset_specific.tallyqa_logger.handlers:
                handler = logging.StreamHandler()
                formatter = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
                handler.setFormatter(formatter)
                evaluate_dataset_specific.tallyqa_logger.addHandler(handler)
        
        logger = evaluate_dataset_specific.tallyqa_logger
        
        # Enhanced ground truth parsing
        gt_num = None
        try:
            if isinstance(gt, str):
                gt_num = int(gt)
            elif isinstance(gt, (int, float)):
                gt_num = int(gt)
            else:
                logger.error(f"Ground truth has unexpected type: {gt} (type: {type(gt).__name__})")
                return False, False
        except (ValueError, TypeError) as e:
            logger.error(f"Failed to parse ground truth: {gt} - Error: {e}")
            return False, False
        
        # Initialize counters for detailed tracking
        if not hasattr(evaluate_dataset_specific, 'tallyqa_main_stats'):
            evaluate_dataset_specific.tallyqa_main_stats = {
                'total_processed': 0,
                'successful_extractions_A': 0,
                'successful_extractions_B': 0,
                'correct_predictions_A': 0,
                'correct_predictions_B': 0
            }
        
        stats = evaluate_dataset_specific.tallyqa_main_stats
        stats['total_processed'] += 1
        
        def extract_number_robust(text: str, prediction_type: str) -> tuple:
            """Extract number with detailed logging."""
            original_text = text
            text = text.strip().lower()
            
            # Method 1: Direct integer parsing
            try:
                result = int(text)
                stats[f'successful_extractions_{prediction_type[-1]}'] += 1
                return result, None
            except ValueError:
                pass
            
            # Method 2: Regex patterns
            patterns = [r'(\d+)\.', r'(\d+)\s', r'(\d+)$', r'(\d+)']
            for pattern in patterns:
                match = re.search(pattern, text)
                if match:
                    try:
                        result = int(match.group(1))
                        stats[f'successful_extractions_{prediction_type[-1]}'] += 1
                        return result, None
                    except ValueError:
                        continue
            
            # Method 3: Word numbers
            word_to_num = {
                'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5,
                'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
            }
            
            for word, num in word_to_num.items():
                if word in text:
                    stats[f'successful_extractions_{prediction_type[-1]}'] += 1
                    return num, None
            
            # Method 4: Negation patterns
            negative_patterns = ['no ', 'none', 'not any', 'cannot see', 'zero']
            for pattern in negative_patterns:
                if pattern in text:
                    stats[f'successful_extractions_{prediction_type[-1]}'] += 1
                    return 0, None
            
            # All methods failed
            error_msg = f"Could not extract number from: '{original_text}'"
            return None, ValueError(error_msg)
        
        # Extract numbers for both predictions
        predA, errorA = extract_number_robust(outA, "Prediction_A")
        predB, errorB = extract_number_robust(outB, "Prediction_B")
        
        # Evaluate correctness
        correctA = False
        correctB = False
        
        if predA is not None:
            correctA = (predA == gt_num)
            if correctA:
                stats['correct_predictions_A'] += 1
        
        if predB is not None:
            correctB = (predB == gt_num)
            if correctB:
                stats['correct_predictions_B'] += 1
        
        return correctA, correctB

    elif dataset == 'refcoco':
        # RefCOCO IoU evaluation
        try:
            pred_bboxA = parse_bbox(outA)
            pred_bboxB = parse_bbox(outB)
            
            if pred_bboxA is None or pred_bboxB is None:
                return False, False
            
            gt_bbox = gt if isinstance(gt, list) else [0, 0, 1, 1]  # Fallback
            
            iouA = compute_iou(pred_bboxA, gt_bbox)
            iouB = compute_iou(pred_bboxB, gt_bbox)
            
            return iouA > 0.5, iouB > 0.5
        except:
            return False, False

    elif dataset == 'textvqa':
        # TextVQA exact match evaluation
        predA_norm = normalize_ans(outA)
        predB_norm = normalize_ans(outB)
        
        if isinstance(gt, list):
            # Multiple acceptable answers
            gt_normalized = [normalize_ans(str(ans)) for ans in gt]
            correctA = predA_norm in gt_normalized
            correctB = predB_norm in gt_normalized
        else:
            gt_norm = normalize_ans(str(gt))
            correctA = predA_norm == gt_norm
            correctB = predB_norm == gt_norm
        
        return correctA, correctB

    elif dataset == 'vqav2':
        # VQAv2 exact match evaluation
        predA_norm = normalize_ans(outA)
        predB_norm = normalize_ans(outB)
        gt_norm = normalize_ans(str(gt))

        correctA = predA_norm == gt_norm
        correctB = predB_norm == gt_norm

        return correctA, correctB

    else:
        # Fallback: exact match
        predA_norm = normalize_ans(outA)
        predB_norm = normalize_ans(outB)
        gt_norm = normalize_ans(str(gt))
        
        return predA_norm == gt_norm, predB_norm == gt_norm

print("Metrics evaluation system ready!")


In [ ]:
# Compute final metrics and results summary

print("Computing final metrics for main entity A/B testing...")
print("="*80)

# Compute metrics per intervention and dataset
metrics = {}

for intervention_name, results in per_intervention_results.items():
    if not results:
        continue
        
    print(f"\nProcessing metrics for {intervention_name}...")
    
    # Group by dataset
    dataset_results = {}
    for result in results:
        dataset = result['dataset']
        if dataset not in dataset_results:
            dataset_results[dataset] = []
        dataset_results[dataset].append(result)
    
    metrics[intervention_name] = {}
    
    for dataset, dataset_items in dataset_results.items():
        print(f"  {dataset}: {len(dataset_items)} examples")
        
        correct_A = 0
        correct_B = 0
        total = len(dataset_items)
        
        for item in dataset_items:
            correctA, correctB = evaluate_dataset_specific(
                dataset, item["gt"], item["outA"], item["outB"]
            )
            if correctA:
                correct_A += 1
            if correctB:
                correct_B += 1
        
        # Calculate metrics
        acc_A = correct_A / total if total > 0 else 0
        acc_B = correct_B / total if total > 0 else 0
        delta_A_minus_B = acc_A - acc_B
        
        metrics[intervention_name][dataset] = {
            "n": total,
            "accA": acc_A,
            "accB": acc_B,
            "delta_A_minus_B": delta_A_minus_B,
        }

print("\n" + "="*80)
print("MAIN ENTITY A/B TESTING RESULTS")
print("="*80)

# Display results in a formatted table
print(f"{'Intervention':<20} {'Dataset':<10} {'N':<6} {'Acc A':<8} {'Acc B':<8} {'Δ(A-B)':<10}")
print("-" * 80)

for intervention_name in sorted(metrics.keys()):
    for dataset in sorted(metrics[intervention_name].keys()):
        metric = metrics[intervention_name][dataset]
        print(f"{intervention_name:<20} {dataset:<10} {metric['n']:<6} "
              f"{metric['accA']:<8.3f} {metric['accB']:<8.3f} {metric['delta_A_minus_B']:<+10.3f}")

# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

all_deltas = []
significant_differences = []

for intervention_name, intervention_metrics in metrics.items():
    intervention_deltas = []
    
    for dataset, metric in intervention_metrics.items():
        delta = metric['delta_A_minus_B']
        all_deltas.append(delta)
        intervention_deltas.append(delta)
        
        # Consider significant if |delta| > 0.05 (5 percentage points)
        if abs(delta) > 0.05:
            significant_differences.append({
                'intervention': intervention_name,
                'dataset': dataset,
                'delta': delta,
                'n': metric['n']
            })
    
    avg_delta = sum(intervention_deltas) / len(intervention_deltas) if intervention_deltas else 0
    print(f"{intervention_name}: Average Δ(A-B) = {avg_delta:+.3f}")

print(f"\nOverall average Δ(A-B): {sum(all_deltas)/len(all_deltas):+.3f}")
print(f"Standard deviation: {(sum([(d - sum(all_deltas)/len(all_deltas))**2 for d in all_deltas]) / len(all_deltas))**0.5:.3f}")

if significant_differences:
    print(f"\nSignificant differences (|Δ| > 0.05):")
    for diff in sorted(significant_differences, key=lambda x: abs(x['delta']), reverse=True):
        print(f"  {diff['intervention']} on {diff['dataset']}: Δ={diff['delta']:+.3f} (n={diff['n']})")
else:
    print(f"\nNo significant differences found (all |Δ| ≤ 0.05)")

print("\n" + "="*60)
print("MAIN ENTITY IDENTIFICATION SUMMARY")
print("="*60)

# Show some examples of main entities identified
print("Sample main entities identified:")
sample_count = 0
for intervention_name, results in per_intervention_results.items():
    if sample_count >= 10:
        break
    for result in results[:3]:  # Show first 3 examples per intervention
        if sample_count >= 10:
            break
        print(f"  {result['dataset']}: \"{result['textA']}\" -> Main entity: '{result['main_entity']}'")
        sample_count += 1

print(f"\nMain entity tokenization A/B testing complete!")
print(f"Analyzed {len(metrics)} interventions across {len(set(d for m in metrics.values() for d in m.keys()))} datasets")
print(f"Total examples evaluated: {sum(sum(m['n'] for m in intervention_metrics.values()) for intervention_metrics in metrics.values())}")


In [ ]:
# Enhanced tokenization logging and verification
print("="*80)
print("TOKENIZATION VERIFICATION AND LOGGING")
print("="*80)

def log_tokenization_details(text_a: str, text_b: str, entity: str, intervention: str):
    """Log detailed tokenization information for verification."""
    
    # Get tokenization analysis
    success, orig_tokens, mod_tokens, details, tokenization_info = check_tokenization_change(text_a, text_b, entity)
    
    print(f"\n--- {intervention.upper()} INTERVENTION ---")
    print(f"Entity: '{entity}'")
    print(f"Original: '{text_a}'")
    print(f"Modified: '{text_b}'")
    print(f"Success: {success}")
    print(f"Token change: +{tokenization_info['token_change']}")
    
    print(f"\nDetailed tokenization:")
    print(f"  Original ({tokenization_info['original_token_count']} tokens):")
    for i, token in enumerate(tokenization_info['original_tokens']):
        print(f"    [{i}] '{token}'")
    
    print(f"  Modified ({tokenization_info['modified_token_count']} tokens):")
    for i, token in enumerate(tokenization_info['modified_tokens']):
        print(f"    [{i}] '{token}'")
    
    print(f"\nReconstructed text verification:")
    print(f"  Original reconstructed: '{tokenization_info['original_reconstructed']}'")
    print(f"  Modified reconstructed: '{tokenization_info['modified_reconstructed']}'")
    print(f"  Original matches: {text_a == tokenization_info['original_reconstructed']}")
    print(f"  Modified matches: {text_b == tokenization_info['modified_reconstructed']}")
    
    print(f"\nEntity analysis:")
    print(f"  Entity '{entity}' in isolation: {tokenization_info['entity_isolated_tokens']} tokens")
    isolated_tokens = tokenizer.encode(entity, add_special_tokens=False)
    isolated_decoded = [tokenizer.decode([tid]) for tid in isolated_tokens]
    print(f"  Entity tokens: {isolated_decoded}")
    
    return tokenization_info

# Test with representative examples if we have data
if 'per_intervention' in locals() and per_intervention:
    print("\\nTesting with collected data samples...")
    
    # Show detailed tokenization for first successful example of each intervention
    shown_count = 0
    for intervention_name, items in per_intervention.items():
        if items and shown_count < 3:  # Limit to first 3 interventions
            sample = items[0]
            log_tokenization_details(
                sample['textA'], 
                sample['textB'], 
                sample['main_entity'], 
                intervention_name
            )
            shown_count += 1
else:
    print("\\nNo collected data available yet. Run data collection first.")

print("\\n" + "="*80)
print("TOKENIZATION LOGGING COMPLETE")
print("="*80)


In [ ]:
# Create comprehensive output files for review (adapted for main entity approach)
import json
import os
from pathlib import Path

# Create interventions directory structure for single entity results
INTERVENTIONS_DIR = Path("/localdisk/ssrivas9/vlm-evaluation/interventions_single_entity")
INTERVENTIONS_DIR.mkdir(exist_ok=True)

print(f"Creating output files in {INTERVENTIONS_DIR}")

# For each intervention, create a subfolder with dataset-specific files
for intervention_name, results in per_intervention_results.items():
    intervention_dir = INTERVENTIONS_DIR / intervention_name
    intervention_dir.mkdir(exist_ok=True)
    
    # Group results by dataset
    dataset_results = {}
    for result in results:
        dataset = result['dataset']
        if dataset not in dataset_results:
            dataset_results[dataset] = []
        dataset_results[dataset].append(result)
    
    # Save each dataset's results
    for dataset, dataset_items in dataset_results.items():
        # Create detailed JSON file
        json_file = intervention_dir / f"{dataset}.json"
        
        # Prepare data for JSON serialization
        json_data = {
            "intervention": intervention_name,
            "dataset": dataset,
            "approach": "main_entity_only",
            "total_examples": len(dataset_items),
            "examples": []
        }
        
        for item in dataset_items:
            # Calculate correctness for this item
            correctA, correctB = evaluate_dataset_specific(dataset, item["gt"], item["outA"], item["outB"])
            
            # Get tokenization info if available
            tokenization_info = item.get('tokenization_info', {})
            
            json_data["examples"].append({
                "id": item["id"],
                "main_entity": item["main_entity"],
                "question_A": item["textA"],
                "question_B": item["textB"],
                "ground_truth": item["gt"],
                "prediction_A": item["outA"],
                "prediction_B": item["outB"],
                "correct_A": correctA,
                "correct_B": correctB,
                "image_path": str(item.get("img_path", "N/A")),
                "tokenization": {
                    "token_change": item.get("token_change", 0),
                    "original_token_count": tokenization_info.get('original_token_count', 0),
                    "modified_token_count": tokenization_info.get('modified_token_count', 0),
                    "original_tokens": tokenization_info.get('original_tokens', []),
                    "modified_tokens": tokenization_info.get('modified_tokens', []),
                    "original_reconstructed": tokenization_info.get('original_reconstructed', ''),
                    "modified_reconstructed": tokenization_info.get('modified_reconstructed', ''),
                    "entity_isolated_tokens": tokenization_info.get('entity_isolated_tokens', 0)
                }
            })
        
        # Save JSON file
        with open(json_file, 'w') as f:
            json.dump(json_data, f, indent=2)
        
        # Create human-readable text file
        txt_file = intervention_dir / f"{dataset}.txt"
        with open(txt_file, 'w') as f:
            f.write(f"Intervention: {intervention_name}\n")
            f.write(f"Dataset: {dataset}\n")
            f.write(f"Approach: Main Entity Only\n")
            f.write(f"Total examples: {len(dataset_items)}\n")
            f.write("=" * 80 + "\n\n")
            
            for i, item in enumerate(dataset_items, 1):
                correctA, correctB = evaluate_dataset_specific(dataset, item["gt"], item["outA"], item["outB"])
                
                f.write(f"Example {i} (ID: {item['id']}):\n")
                f.write(f"  Main entity modified: '{item['main_entity']}'\n")
                f.write(f"  Question A: {item['textA']}\n")
                f.write(f"  Question B: {item['textB']}\n")
                f.write(f"  Ground Truth: {item['gt']}\n")
                f.write(f"  Prediction A: {item['outA']} ({'✓' if correctA else '✗'})\n")
                f.write(f"  Prediction B: {item['outB']} ({'✓' if correctB else '✗'})\n")
                f.write(f"  Image: {item.get('img_path', 'N/A')}\n")
                f.write("-" * 40 + "\n\n")
    
    print(f"  {intervention_name}: {len(dataset_results)} datasets saved")

# Create summary file
summary_file = INTERVENTIONS_DIR / "summary.json"
summary_data = {
    "approach": "main_entity_tokenization",
    "description": "A/B testing focusing only on the main entity central to each question",
    "total_interventions": len(per_intervention_results),
    "total_examples": sum(len(results) for results in per_intervention_results.values()),
    "interventions": {},
    "metrics": {}
}

# Add intervention summaries
for intervention_name, results in per_intervention_results.items():
    dataset_breakdown = {}
    for result in results:
        dataset = result['dataset']
        dataset_breakdown[dataset] = dataset_breakdown.get(dataset, 0) + 1
    
    summary_data["interventions"][intervention_name] = {
        "total_examples": len(results),
        "datasets": dataset_breakdown
    }

# Add metrics
for intervention_name, dataset_metrics in metrics.items():
    summary_data["metrics"][intervention_name] = {}
    for dataset, metric in dataset_metrics.items():
        summary_data["metrics"][intervention_name][dataset] = {
            "n": metric["n"],
            "accuracy_A": round(metric["accA"], 3),
            "accuracy_B": round(metric["accB"], 3),
            "delta_A_minus_B": round(metric["delta_A_minus_B"], 3)
        }

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\nOutput files created:")
print(f"  - Summary: {summary_file}")
for intervention_name in per_intervention_results.keys():
    intervention_dir = INTERVENTIONS_DIR / intervention_name
    files = list(intervention_dir.glob("*"))
    print(f"  - {intervention_name}: {len(files)} files in {intervention_dir}")

print(f"\nTotal files created: {len(list(INTERVENTIONS_DIR.rglob('*')))} files")
print(f"Directory structure: /interventions_single_entity/[intervention_name]/[dataset].[json|txt]")

# Show comparison with multi-entity approach
print(f"\n" + "="*60)
print("COMPARISON WITH MULTI-ENTITY APPROACH")
print("="*60)
print("This notebook (main entity): /interventions_single_entity/")
print("Multi-entity notebook:       /interventions/")
print("\nKey differences in output files:")
print("- main_entity field instead of entities/entity_count")
print("- approach field indicates 'main_entity_only'")
print("- focuses on single most important entity per question")
print("- uses advanced NLP techniques for entity identification")
